In [1]:
# Import the packages we need

import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
file_path = "data.csv"
df = pd.read_csv(file_path)

# Check the loaded data
print("Dataset shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
display(df.head())

# Separate the 86 permission features from the target label
X = df.drop(columns="Result")
y = df["Result"]

print("Number of features:", X.shape[1])
print("Number of observations:", X.shape[0])
print("\nClass counts:")
print(y.value_counts())

# Create a group label for each unique permission pattern
permission_groups = X.astype(str).agg("_".join, axis=1)

print(
    "\nNumber of unique permission patterns:",
    permission_groups.nunique()
)

# Count apps that share each permission pattern
group_counts = permission_groups.value_counts()

print("\nMost common permission patterns:")
print(group_counts.head(10))

# Count rows identical across all features and the target
print("\nNumber of exact duplicate rows:", df.duplicated().sum())

# Count extra apps beyond the first for each permission pattern
print(
    "Number of repeated permission-pattern rows:",
    len(permission_groups) - permission_groups.nunique()
)

# Check whether any permission pattern has both labels
label_counts = y.groupby(permission_groups).nunique()
conflicting_groups = label_counts[label_counts > 1]

print(
    "\nNumber of permission patterns with BOTH labels:",
    len(conflicting_groups)
)

# Create one row per permission pattern for group-based splitting.
# Mark patterns that have both labels as -1.
group_labels = y.groupby(permission_groups, sort=False).agg(
    lambda labels: labels.iloc[0] if labels.nunique() == 1 else -1
)

group_data = X.groupby(permission_groups, sort=False).first()
group_data["Result"] = group_labels

print("\nGroup-level label counts:")
print(group_data["Result"].value_counts())
print("Group-level dataset shape:", group_data.shape)

# Split unique permission patterns into training (70%) and testing (30%)
train_groups, test_groups = train_test_split(
    group_labels.index,
    test_size=0.30,
    random_state=42
)

# Put each original app in the split assigned to its permission pattern
train_mask = permission_groups.isin(train_groups)
test_mask = permission_groups.isin(test_groups)

train_data = df.loc[train_mask].copy()
test_data = df.loc[test_mask].copy()

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
print(
    "Permission patterns shared between splits:",
    len(set(train_groups) & set(test_groups))
)

print("blah blah blah")

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'